# MediFlow Hair — 256 모델 오답·라벨 구조 점검

이 노트북은 학습을 다시 하지 않습니다. 선택된 256×256 Hair 모델로 정리 데이터의 Test 세트만 재현하고 사람이 확인할 오답 자료를 만듭니다.

- 핵심 대상: 미세각질 ↔ 비듬, 비듬 ↔ 피지과다
- 저장 내용: 전체 예측 CSV, 오답 CSV, 혼동 쌍 요약, 검토용 이미지, 이미지 시트
- 라벨 점검: split_manifest.csv를 연결하여 원래 파일 경로와 SHA-256 보존
- 보호 원칙: 데이터 ZIP과 모델은 읽기만 하며 기존 파일을 덮어쓰지 않음
- 한계: 원천 JSON이 clean ZIP에 없다면 다중 라벨 여부를 확정하지 않음

Test 결과는 오류 유형 확인에만 사용하며 같은 Test를 보고 설정을 반복 변경하지 않습니다.


## 1. Colab GPU와 Drive 준비

GPU 런타임을 선택한 후 모두 실행하세요. Drive 연결 창이 나타나면 승인합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install seaborn scikit-learn

import hashlib
import json
import math
import platform
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU가 없습니다. Colab 런타임 유형을 GPU로 변경하세요.')


## 2. 분석 설정

현재 선택된 256 모델과 clean 데이터 경로가 입력되어 있습니다. Drive에서 파일을 옮긴 경우에만 경로를 바꾸세요.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_PATH = MY_DRIVE / 'mediflow_datasets' / 'hair_clean_v1_20260907_053616.zip'
EXPECTED_DATA_SHA256 = '2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156'
MODEL_RESULT_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_two_stage_20260907_064214'
MODEL_PATH = MODEL_RESULT_ROOT / 'best_model.keras'
MODEL_CONFIG_PATH = MODEL_RESULT_ROOT / 'training_config.json'

CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
CLASS_CODES = {name: f'C{index}' for index, name in enumerate(CLASS_NAMES)}
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EXPECTED_TEST_ACCURACY = 0.7891373801916933
EXPECTED_MACRO_F1 = 0.7886477230068456
METRIC_TOLERANCE = 0.001
FOCUS_PAIRS = [('미세각질', '비듬'), ('비듬', '미세각질'), ('비듬', '피지과다'), ('피지과다', '비듬')]
CONTACT_SHEET_LIMIT = 25
SEED = 42

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOCAL_ROOT = Path('/content') / f'hair_error_audit_{RUN_ID}'
LOCAL_ZIP_PATH = LOCAL_ROOT / DATA_ZIP_PATH.name
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
RESULT_ROOT = LOCAL_ROOT / 'results'
REVIEW_ROOT = RESULT_ROOT / 'review_images'
DRIVE_RESULT_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / f'error_audit_256_{RUN_ID}'
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir()
RESULT_ROOT.mkdir()
REVIEW_ROOT.mkdir()
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('분석 ID:', RUN_ID)
print('데이터:', DATA_ZIP_PATH)
print('모델:', MODEL_PATH)
print('결과:', DRIVE_RESULT_ROOT)
print('클래스 코드:', CLASS_CODES)


## 3. 데이터와 모델 계약 검증

ZIP 식별값, 모델 입력·출력, 내부 Rescaling(1/255) 계층과 학습 당시 설정을 확인합니다.


In [ ]:
def copy_with_sha256(source, destination):
    digest = hashlib.sha256()
    with source.open('rb') as src, destination.open('wb') as dst:
        while True:
            chunk = src.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            dst.write(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination_resolved / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
        archive.extractall(destination_resolved)

def flatten_layers(layer):
    found = []
    for child in getattr(layer, 'layers', []):
        found.append(child)
        found.extend(flatten_layers(child))
    return found

for required_path in (DATA_ZIP_PATH, MODEL_PATH, MODEL_CONFIG_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'필수 파일을 찾지 못했습니다: {required_path}')

data_sha256 = copy_with_sha256(DATA_ZIP_PATH, LOCAL_ZIP_PATH)
if data_sha256 != EXPECTED_DATA_SHA256:
    raise ValueError(f'데이터 ZIP 식별값이 다릅니다. 예상={EXPECTED_DATA_SHA256}, 실제={data_sha256}')
safe_extract(LOCAL_ZIP_PATH, EXTRACT_ROOT)

training_config = json.loads(MODEL_CONFIG_PATH.read_text(encoding='utf-8'))
if training_config.get('class_names') != CLASS_NAMES:
    raise ValueError(f"학습 클래스 순서가 다릅니다: {training_config.get('class_names')}")
if training_config.get('data_zip_sha256') != EXPECTED_DATA_SHA256:
    raise ValueError('모델 학습 데이터와 현재 ZIP이 다릅니다.')
if training_config.get('image_size') != list(IMAGE_SIZE):
    raise ValueError(f"학습 입력 크기가 다릅니다: {training_config.get('image_size')}")

model = keras.models.load_model(MODEL_PATH, compile=False)
if tuple(model.input_shape[1:]) != (*IMAGE_SIZE, 3):
    raise ValueError(f'모델 입력이 예상과 다릅니다: {model.input_shape}')
if int(model.output_shape[-1]) != len(CLASS_NAMES):
    raise ValueError(f'모델 출력 수가 예상과 다릅니다: {model.output_shape}')
rescaling_layers = [layer for layer in flatten_layers(model) if isinstance(layer, keras.layers.Rescaling)]
if not rescaling_layers or not any(np.isclose(float(layer.scale), 1 / 255) for layer in rescaling_layers):
    raise ValueError('모델 내부 Rescaling(1/255)을 확인하지 못했습니다.')
print('데이터·모델 계약 검증 완료')
print('입력:', model.input_shape, '출력:', model.output_shape)


## 4. Test 데이터와 원본 이력 연결

증강되지 않은 original/test만 사용합니다. split_manifest.csv가 있으면 새 파일명에 원래 경로와 SHA-256을 연결합니다.


In [ ]:
def find_dataset_root():
    matches = [path for path in EXTRACT_ROOT.rglob('original') if path.is_dir() and all((path / split).is_dir() for split in ('train', 'val', 'test'))]
    if len(matches) != 1:
        raise ValueError(f'original 데이터 폴더를 하나로 확정할 수 없습니다: {matches}')
    return matches[0].parent

DATASET_ROOT = find_dataset_root()
TEST_ROOT = DATASET_ROOT / 'original' / 'test'
found_classes = sorted(path.name for path in TEST_ROOT.iterdir() if path.is_dir())
if found_classes != sorted(CLASS_NAMES):
    raise ValueError(f'Test 클래스 구성이 예상과 다릅니다: {found_classes}')

test_raw = tf.keras.utils.image_dataset_from_directory(TEST_ROOT, labels='inferred', label_mode='int', class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False)
test_paths = [Path(path) for path in test_raw.file_paths]
test_ds = test_raw.prefetch(tf.data.AUTOTUNE)
expected_labels_from_paths = np.array([CLASS_NAMES.index(path.parent.name) for path in test_paths], dtype=np.int64)

manifest_candidates = list(DATASET_ROOT.rglob('split_manifest.csv'))
if len(manifest_candidates) > 1:
    raise ValueError(f'split_manifest.csv가 여러 개입니다: {manifest_candidates}')
split_manifest = pd.read_csv(manifest_candidates[0], encoding='utf-8-sig') if manifest_candidates else pd.DataFrame()
print('Test 이미지 수:', len(test_paths))
print('split_manifest:', manifest_candidates[0] if manifest_candidates else '없음')


## 5. 전체 예측 재현

학습 때와 같은 모델·데이터인지 기존 Test 지표와 비교합니다. 허용 오차를 넘으면 분석을 중단합니다.


In [ ]:
probability_batches, label_batches = [], []
for images, labels in test_ds:
    probability_batches.append(model.predict(images, verbose=0))
    label_batches.append(labels.numpy())

probabilities = np.concatenate(probability_batches, axis=0)
y_true = np.concatenate(label_batches).astype(np.int64)
if not np.array_equal(y_true, expected_labels_from_paths):
    raise ValueError('파일 경로의 클래스와 TensorFlow 라벨 순서가 다릅니다.')
y_pred = np.argmax(probabilities, axis=1)
sorted_indices = np.argsort(probabilities, axis=1)
top2_indices = sorted_indices[:, -2]
top1_probability = probabilities[np.arange(len(probabilities)), y_pred]
top2_probability = probabilities[np.arange(len(probabilities)), top2_indices]
margins = top1_probability - top2_probability

test_accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
if abs(test_accuracy - EXPECTED_TEST_ACCURACY) > METRIC_TOLERANCE:
    raise ValueError(f'Test Accuracy가 기존 결과와 다릅니다: {test_accuracy}')
if abs(macro_f1 - EXPECTED_MACRO_F1) > METRIC_TOLERANCE:
    raise ValueError(f'Macro F1이 기존 결과와 다릅니다: {macro_f1}')

rows = []
for index, path in enumerate(test_paths):
    row = {'case_id': f'case_{index + 1:04d}', 'dataset_path': path.relative_to(DATASET_ROOT).as_posix(), 'file_name': path.name, 'true_index': int(y_true[index]), 'true_class': CLASS_NAMES[y_true[index]], 'pred_index': int(y_pred[index]), 'pred_class': CLASS_NAMES[y_pred[index]], 'correct': bool(y_true[index] == y_pred[index]), 'top1_probability': float(top1_probability[index]), 'top2_class': CLASS_NAMES[top2_indices[index]], 'top2_probability': float(top2_probability[index]), 'top1_top2_margin': float(margins[index])}
    for class_index, class_name in enumerate(CLASS_NAMES):
        row[f'prob_{CLASS_CODES[class_name]}_{class_name}'] = float(probabilities[index, class_index])
    rows.append(row)
predictions_df = pd.DataFrame(rows)

if not split_manifest.empty:
    provenance = split_manifest.copy()
    provenance['dataset_path'] = provenance['output_path'].astype(str).str.replace('\\', '/', regex=False)
    provenance = provenance[['dataset_path', 'sha256', 'old_split', 'source_path']]
    predictions_df = predictions_df.merge(provenance, on='dataset_path', how='left', validate='one_to_one')
    if predictions_df['sha256'].isna().any():
        raise ValueError('원본 이력이 연결되지 않은 Test 파일이 있습니다.')
print('재현 Test Accuracy:', test_accuracy)
print('재현 Macro F1:', macro_f1)
print('기존 결과와 일치: 예')


## 6. 혼동 쌍 요약과 검토 순서

핵심 혼동, 높은 확신 오답, Top1과 Top2 차이가 작은 사례 순으로 사람이 검토할 자료를 만듭니다.


In [ ]:
report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
errors_df = predictions_df[~predictions_df['correct']].copy()

confusion_rows = []
for true_index, true_name in enumerate(CLASS_NAMES):
    for pred_index, pred_name in enumerate(CLASS_NAMES):
        if true_index == pred_index:
            continue
        subset = errors_df[(errors_df['true_class'] == true_name) & (errors_df['pred_class'] == pred_name)]
        confusion_rows.append({'true_class': true_name, 'pred_class': pred_name, 'count': int(len(subset)), 'mean_top1_probability': float(subset['top1_probability'].mean()) if len(subset) else None, 'mean_margin': float(subset['top1_top2_margin'].mean()) if len(subset) else None})
confusion_pairs_df = pd.DataFrame(confusion_rows).sort_values(['count', 'true_class', 'pred_class'], ascending=[False, True, True])

focus_mask = pd.Series(False, index=predictions_df.index)
for true_name, pred_name in FOCUS_PAIRS:
    focus_mask |= (predictions_df['true_class'] == true_name) & (predictions_df['pred_class'] == pred_name)
focus_errors_df = predictions_df[focus_mask & ~predictions_df['correct']].copy()

review_queue = errors_df.copy()
review_queue['focus_pair'] = False
for true_name, pred_name in FOCUS_PAIRS:
    review_queue.loc[(review_queue['true_class'] == true_name) & (review_queue['pred_class'] == pred_name), 'focus_pair'] = True
review_queue = review_queue.sort_values(['focus_pair', 'top1_probability', 'top1_top2_margin'], ascending=[False, False, True]).reset_index(drop=True)
review_queue['review_order'] = np.arange(1, len(review_queue) + 1)
review_queue['review_status'] = ''
review_queue['review_note'] = ''

for _, row in review_queue.iterrows():
    source = DATASET_ROOT / row['dataset_path']
    pair_folder = f"{CLASS_CODES[row['true_class']]}_true__{CLASS_CODES[row['pred_class']]}_pred"
    destination_dir = REVIEW_ROOT / pair_folder
    destination_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination_dir / f"{row['case_id']}{source.suffix.lower()}")

high_confidence_errors = errors_df.sort_values('top1_probability', ascending=False).head(100)
low_margin_cases = predictions_df.sort_values('top1_top2_margin', ascending=True).head(100)
display(confusion_pairs_df.head(12))
print('전체 오답:', len(errors_df))
print('핵심 혼동 오답:', len(focus_errors_df))


## 7. 검토용 이미지 시트 생성

표기는 case ID / 실제 코드→예측 코드 / 확률 / margin입니다. 클래스 코드 표는 마지막 셀에도 저장됩니다.


In [ ]:
def make_contact_sheet(frame, output_path, title, limit=CONTACT_SHEET_LIMIT):
    frame = frame.head(limit).copy()
    if frame.empty:
        return
    thumb_width, thumb_height, label_height, columns = 220, 220, 58, 5
    rows = math.ceil(len(frame) / columns)
    canvas = Image.new('RGB', (columns * thumb_width, 48 + rows * (thumb_height + label_height)), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 12), title, fill='black')
    for position, (_, row) in enumerate(frame.iterrows()):
        source = DATASET_ROOT / row['dataset_path']
        with Image.open(source) as image:
            image = ImageOps.exif_transpose(image).convert('RGB')
            image.thumbnail((thumb_width - 10, thumb_height - 10))
            tile = Image.new('RGB', (thumb_width, thumb_height), '#dddddd')
            tile.paste(image, ((thumb_width - image.width) // 2, (thumb_height - image.height) // 2))
        column, row_number = position % columns, position // columns
        x, y = column * thumb_width, 48 + row_number * (thumb_height + label_height)
        canvas.paste(tile, (x, y))
        draw.text((x + 5, y + thumb_height + 5), f"{row['case_id']}  {CLASS_CODES[row['true_class']]}>{CLASS_CODES[row['pred_class']]}", fill='black')
        draw.text((x + 5, y + thumb_height + 25), f"p={row['top1_probability']:.3f}  margin={row['top1_top2_margin']:.3f}", fill='black')
    canvas.save(output_path, quality=92)

for true_name, pred_name in FOCUS_PAIRS:
    pair_frame = errors_df[(errors_df['true_class'] == true_name) & (errors_df['pred_class'] == pred_name)].sort_values('top1_probability', ascending=False)
    make_contact_sheet(pair_frame, RESULT_ROOT / f"focus_{CLASS_CODES[true_name]}_to_{CLASS_CODES[pred_name]}.jpg", f"{CLASS_CODES[true_name]} true -> {CLASS_CODES[pred_name]} predicted")
make_contact_sheet(high_confidence_errors, RESULT_ROOT / 'high_confidence_errors.jpg', 'Highest-confidence wrong predictions')
make_contact_sheet(low_margin_cases, RESULT_ROOT / 'lowest_margin_cases.jpg', 'Smallest Top1-Top2 margins')
print('이미지 시트 생성 완료')


## 8. 그래프와 라벨 구조 점검

clean ZIP 안에 원천 JSON이 있는지 확인합니다. JSON이 없으면 다중 라벨 여부는 확인 불가로 기록합니다.


In [ ]:
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Hair 256 Error Audit')
plt.tight_layout(); plt.savefig(RESULT_ROOT / 'confusion_matrix_reproduced.png', dpi=200, bbox_inches='tight'); plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(data=predictions_df, x='top1_top2_margin', hue='correct', bins=20, multiple='layer')
plt.xlabel('Top1 - Top2 probability margin'); plt.title('Prediction margin distribution')
plt.tight_layout(); plt.savefig(RESULT_ROOT / 'margin_distribution.png', dpi=200, bbox_inches='tight'); plt.show()

json_files = [path for path in EXTRACT_ROOT.rglob('*.json') if path.name != 'dataset_summary.json']
label_structure_audit = {
    'source_json_count_in_clean_zip': len(json_files),
    'source_json_examples': [str(path.relative_to(EXTRACT_ROOT)) for path in json_files[:20]],
    'multilabel_status': '원천 JSON 발견: 별도 스키마 분석 필요' if json_files else '확인 불가: clean ZIP에 원천 라벨 JSON이 없음',
    'inference_not_a_fact': '동일 이미지에 서로 다른 클래스가 붙었던 현상은 다중 증상 가능성을 시사하지만 원천 JSON 확인 전에는 확정할 수 없음',
}
print(json.dumps(label_structure_audit, ensure_ascii=False, indent=2))


## 9. 결과 저장

기존 모델과 데이터는 그대로 두고 새 Drive 폴더와 ZIP을 만듭니다. ZIP에는 모델 파일을 넣지 않습니다.


In [ ]:
predictions_df.to_csv(RESULT_ROOT / 'all_test_predictions.csv', index=False, encoding='utf-8-sig')
errors_df.to_csv(RESULT_ROOT / 'all_misclassifications.csv', index=False, encoding='utf-8-sig')
focus_errors_df.to_csv(RESULT_ROOT / 'focus_pair_errors.csv', index=False, encoding='utf-8-sig')
confusion_pairs_df.to_csv(RESULT_ROOT / 'confusion_pairs_summary.csv', index=False, encoding='utf-8-sig')
review_queue.to_csv(RESULT_ROOT / 'manual_review_queue.csv', index=False, encoding='utf-8-sig')
high_confidence_errors.to_csv(RESULT_ROOT / 'high_confidence_errors.csv', index=False, encoding='utf-8-sig')
low_margin_cases.to_csv(RESULT_ROOT / 'lowest_margin_cases.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(report).transpose().to_csv(RESULT_ROOT / 'classification_report_reproduced.csv', encoding='utf-8-sig')
(RESULT_ROOT / 'label_structure_audit.json').write_text(json.dumps(label_structure_audit, ensure_ascii=False, indent=2), encoding='utf-8')

audit_summary = {
    'audit_id': RUN_ID, 'model_path': str(MODEL_PATH), 'data_zip_path': str(DATA_ZIP_PATH),
    'data_zip_sha256': data_sha256, 'class_names': CLASS_NAMES, 'class_codes': CLASS_CODES,
    'image_size': list(IMAGE_SIZE), 'test_image_count': int(len(predictions_df)),
    'test_accuracy_reproduced': test_accuracy, 'macro_f1_reproduced': macro_f1,
    'misclassification_count': int(len(errors_df)), 'focus_pair_error_count': int(len(focus_errors_df)),
    'focus_pairs': [list(pair) for pair in FOCUS_PAIRS],
    'selection_policy': '오답 분석은 오류 유형 확인용이며 Test 수치로 같은 데이터의 설정을 반복 선택하지 않음',
    'limitations': ['실제 USB 현미경 환자 데이터가 없어 Domain Gap을 평가하지 못함', '사람·촬영 세션 식별자가 없어 해당 단위의 분할 누수는 확인하지 못함', label_structure_audit['multilabel_status']],
}
(RESULT_ROOT / 'audit_summary.json').write_text(json.dumps(audit_summary, ensure_ascii=False, indent=2), encoding='utf-8')

readme = f"""# Hair 256 error audit

- Test Accuracy 재현: {test_accuracy:.6f}
- Macro F1 재현: {macro_f1:.6f}
- 전체 오답: {len(errors_df)}
- 핵심 혼동 오답: {len(focus_errors_df)}
- 다중 라벨 확인 상태: {label_structure_audit['multilabel_status']}

manual_review_queue.csv의 review_status에는 label_correct, ambiguous, wrong_label, unusable 중 하나를 입력합니다.
"""
(RESULT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

if DRIVE_RESULT_ROOT.exists():
    raise FileExistsError(f'결과 폴더가 이미 있습니다. 덮어쓰지 않습니다: {DRIVE_RESULT_ROOT}')
DRIVE_RESULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RESULT_ROOT, DRIVE_RESULT_ROOT)
drive_zip_path = Path(shutil.make_archive(str(DRIVE_RESULT_ROOT), 'zip', root_dir=DRIVE_RESULT_ROOT.parent, base_dir=DRIVE_RESULT_ROOT.name))
print('Drive 분석 폴더:', DRIVE_RESULT_ROOT)
print('Drive 분석 ZIP:', drive_zip_path)
print('검토표:', DRIVE_RESULT_ROOT / 'manual_review_queue.csv')
print('클래스 코드:', CLASS_CODES)


## 완료 후 보내줄 것

마지막 셀의 Drive 분석 ZIP을 다운로드해서 보내주세요. ZIP이 크면 audit_summary.json, confusion_pairs_summary.csv, manual_review_queue.csv와 JPG 이미지 시트들만 보내도 됩니다.

클래스 코드는 C0=모낭사이홍반, C1=미세각질, C2=비듬, C3=탈모, C4=피지과다입니다.
